## **6. Key-Value RDD 연산**

### **Spark 애플리케이션 시작**

In [1]:
# [+] PySpark 임포트 및 설정
from pyspark import SparkConf, SparkContext

conf = SparkConf().setMaster('local').setAppName("reducation-operations")
sc = SparkContext(conf=conf)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/08 04:44:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### **groupBy( )**
입력 함수를 기준으로 그룹핑

In [2]:
# [+] 리스트로부터 rdd 생성 [1, 1, 2, 3, 5, 8]
rdd = sc.parallelize([1,1,2,3,5,8])

In [8]:
# [+] groupBy()를 이용한 홀수/짝수 그룹핑 -> groups
groups = rdd.groupBy(lambda x : x % 2)

In [9]:
# [+] 그룹핑 결과를 리스트로 반환 -> res
res = groups.collect()

In [10]:
# [+] 그룹핑 결과(res) 출력
for k, v in res:
    print(k, list(v))

1 [1, 1, 3, 5]
0 [2, 8]


### **groupByKey( )**
Key 값을 기준으로 그룹핑

In [11]:
# [+] 리스트로부터 key-value rdd 생성 [('a', 1), ('b', 1), ('a', 1)]
rdd = sc.parallelize([("a", 1), ("b", 1), ("a", 1)])
rdd.collect()

[('a', 1), ('b', 1), ('a', 1)]

In [12]:
rdd.keys().collect()

['a', 'b', 'a']

In [13]:
# [+] groupByKey()를 이용한 그룹핑 및 리스트 반환
groups = rdd.groupByKey()

In [14]:
# [+] 그룹핑 결과를 리스트로 반환
res = groups.collect()

In [15]:
# [+]그룹핑 결과(res) 출력
for k, v in res:
    print(k, list(v))

a [1, 1]
b [1]


In [16]:
# len(): Python 객체의 길이를 리턴
value = [1, 1]
len(value)

2

In [17]:
# mapValues(), len()을 이용한 값 빈도 계산
counts = groups.mapValues(len)

In [18]:
counts.collect()

[('a', 2), ('b', 1)]

### **groupBy( ) 예제**
1. 리스트로부터 RDD 객체 생성
    + 리스트: ['C', 'C++', 'Python', 'Java', 'C#']  
2. 원소의 첫 번째 문자 값을 기준으로 그룹핑
    + 기대 결과:  
        J ['Java']  
        C ['C', 'C++', 'C#']  
        P ['Python']  
3. 그룹핑 결과를 리스트 객체로 출력

In [19]:
# [+] 리스트로부터 RDD 객체 생성
rdd = sc.parallelize(['C', 'C++', 'Python', 'Java', 'C#']  )

In [20]:
# [+] 원소별 첫 번째 값(x[0])을 기준으로 그룹핑 -> groups
groups = rdd.groupBy(lambda x : x[0])

In [21]:
# [+] 그룹핑 결과를 리스트로 반환 -> res
res = groups.collect()

In [22]:
# [+] 그룹핑 결과(res) 출력
for k, v in res:
    print(k, list(v))

C ['C', 'C++', 'C#']
P ['Python']
J ['Java']


### **groupByKey( ) 예제**
1. list 객체로부터 3개의 파티션을 갖는 Key-Value RDD 생성하기
    + ```getNumPatitions()```: RDD 객체의 파티션 수 출력
2. Key 별로 그룹핑
3. 그룹핑 결과 리스트로 반환
4. 리스트 출력

In [23]:
# list 객체로부터 3개의 파티션을 갖는 Key-Value RDD 생성하기

rdd = sc.parallelize([  # 과목 별 점수 데이터
    ('Math', 7), ('Math', 2), ('English', 7),
    ('Science', 7), ('English', 4), ('English', 9),
    ('Math', 8), ('Math', 3), ('English', 4),
    ('Science', 6), ('Science', 9), ('Science', 5)
], 3) # 파티션 수: 3

In [24]:
# getNumPartitions(): RDD 객체의 파티션 수를 확인하는 액션 메서드
rdd.getNumPartitions()

3

In [25]:
# [+] 파티션별 RDD 값 확인
rdd.glom().collect()

[[('Math', 7), ('Math', 2), ('English', 7), ('Science', 7)],
 [('English', 4), ('English', 9), ('Math', 8), ('Math', 3)],
 [('English', 4), ('Science', 6), ('Science', 9), ('Science', 5)]]

In [27]:
# [+] Key 별로 그룹핑 -> groups
groups = rdd.groupByKey()


In [28]:
# [+] 그룹핑 결과 리스트로 반환 -> res
res = groups.collect()

In [29]:
# 그룹핑 결과(res) 출력
for k, v in res:
    print(k, list(v))

Math [7, 2, 8, 3]
Science [7, 6, 9, 5]
English [7, 4, 9, 4]


In [30]:
# key별 그룹핑을 2개의 파티션으로 분리하여 수행 -> groups
groups = rdd.groupByKey(2)

In [31]:
# [+] 파티션 수 출력
groups.getNumPartitions()

2

In [32]:
# [+] 파티션별 RDD 그룹 확인
res = groups.collect()
res

[('Science', <pyspark.resultiterable.ResultIterable at 0x7b3a7c847dd0>),
 ('Math', <pyspark.resultiterable.ResultIterable at 0x7b3a7c844890>),
 ('English', <pyspark.resultiterable.ResultIterable at 0x7b3a7c847650>)]

In [34]:
# [+] 0번째 파티션에 대한 그룹핑 결과(res)를 출력
print(res[0][0][0], list(res[0][0][1]))

S ['c']


In [39]:
# [+] 1번째 파티션에 대한 그룹핑 결과(res)를 출력
for k, v in res[1]:
    print(k, list(v))

ValueError: too many values to unpack (expected 2)

### **reduce( ) vs reduceByKey( )**
+ ```reduce()```: 입력 함수를 기준으로 집계를 수행하는 액션 메서드
+ ```reduceByKey()```: Key를 기준으로 그룹핑 및 집계를 수행하는 변환 메서드

In [ ]:
# [+] reduce() 를 이용한 총합 구하기
sc.parallelize([1, 2, 3, 4, 5])...

In [ ]:
# 리스트로부터 K-V RDD 생성 -> rdd
rdd = sc.parallelize([('a', 1), ('b', 1), ('a', 1)])

In [ ]:
# [+] reduceByKey()를 이용한 Key별 총합 계산 및 리스트로 반환
rdd...

In [ ]:
# 과목 별 점수 데이터
rdd = sc.parallelize([
    ('Math', 7), ('Math', 2), ('English', 7),
    ('Science', 7), ('English', 4), ('English', 9),
    ('Math', 8), ('Math', 3), ('English', 4),
    ('Science', 6), ('Science', 9), ('Science', 5)
], 3)

In [ ]:
# [+] reduceByKey() 를 이용한 과목 별 점수 총합 구하기
rdd...

### **mapValues( ) vs map( )**
+ ```map()```: key, value 전부 함수에 적용
+ ```mapValues()```: value에만 함수를 적용

In [ ]:
# 리스트로부터 K-V RDD 객체 생성
rdd = sc.parallelize([
    ('a', ['apple', 'banana', 'lemon']),
    ('b', ['grapes'])
])

In [ ]:
# mapValues(), len()를 이용하여 키 별 갯수 세기
rdd.mapValues(len).collect()

In [ ]:
# map(): key, value 에 함수를 적용
rdd.map(len).collect()  # 결과가 mapValues()와 다른 이유는?

### **countByKey( )**
Key별 값 개수를 계산 및 딕셔너리로 반환하는 액션 메서드

In [ ]:
# countByKey()를 이용하여 Key별 값 배수 계산 및 반환
rdd = sc.parallelize([
    ('a', 1), ('b', 1), ('a', 1)
])

res = rdd.countByKey()

In [ ]:
# countByKey() 결과 출력
print(res)

### **keys( )**
key 목록을 RDD로 생성하는 변환 메서드

In [ ]:
# 과목 별 점수 데이터
rdd = sc.parallelize([
    ('Math', 7), ('Math', 2), ('English', 7),
    ('Science', 7), ('English', 4), ('English', 9),
    ('Math', 8), ('Math', 3), ('English', 4),
    ('Science', 6), ('Science', 9), ('Science', 5)
], 3)



In [ ]:
# RDD 키 목록 생성 및 출력
rdd.keys().collect()

In [ ]:
# [+] 유니크한 키 값 출력
rdd...

In [ ]:
# [+] 유니크한 키 개수 출력
rdd...

### **Join 연산**
+ ```join()```: 내부조인
+ ```leftOuterJoin()```: 외부조인(첫 번째 RDD 중심)
+ ```rightOuterJoin()```: 외부조인(두 번째 RDD 중심)

In [ ]:
# K-V RDD 객체 두 개 생성 
rdd1 = sc.parallelize([("a", 1), ("b", 4)])
rdd2 = sc.parallelize([("a", 2)])

In [ ]:
# Join 연산
rdd1.join(rdd2).collect()

In [ ]:
# Left Outer Join 연산
rdd1.leftOuterJoin(rdd2).collect()

In [ ]:
# Right Outer Join 연산
rdd1.rightOuterJoin(rdd2).collect()